# **Driver Advisory LLM — ZMQ Receiver + RAG**

**Pipeline:**
1. Receives 30-second aggregated payloads from `inference.ipynb` via ZMQ (port 5555)
2. Skips the LLM entirely if the driver is fully alert (all-zero payload)
3. Translates the payload into a natural-language narrative
4. **[RAG]** Selects the 2 most relevant knowledge chunks from `rag_knowledge_base.json`
5. **[RAG]** Injects those chunks into the prompt as grounding context
6. Feeds the enriched prompt into the local LLM (`llama-cpp-python`)
7. Outputs structured JSON: `{driver_state, risk_level, message}` — `message` is ready for TTS

**Token budget note:** Small models (SmolLM2-135M, TinyLlama) have a 512-token context.
RAG chunks average ~60-80 tokens each. This notebook injects a max of **2 chunks** to stay
safely within the window. If you switch to Mistral-7B (N_CTX=2048+), raise `RAG_TOP_K` to 4-5.

**Run order:** Start this notebook first, then run `inference.ipynb`.

## Cell 1 — Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install pyzmq llama-cpp-python --quiet


[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: pip3.11 install --upgrade pip


## Cell 2 — Configuration

In [2]:
# ZMQ CONFIG 
ZMQ_PORT = 5555                          # Must match inference.ipynb!!!

# MODEL CONFIG IF NEED
# MODEL_PATH   = "Models/SmolLM2-135M-Instruct-Q4_K_M.gguf"   # fastest
# MODEL_PATH = "../Models/Qwen2.5-0.5B-Instruct-Q4_K_M.gguf" # Test run
MODEL_PATH = "../Models/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf" # Test run

# N_CTX        = 512    # fastest
N_CTX = 2048  # Qwen2.5 handles larger context well
MAX_TOKENS   = 80
TEMPERATURE  = 0.3

# RAG CONFIG
RAG_PATH  = "../llm/rag_knowledge_base.json"   # Path to your knowledge base file
RAG_TOP_K = 2   # Max chunks to inject. Keep at 2 for small models (512 ctx).
                # Raise to 4-5 if using Mistral-7B with larger context window.

# ALERT THRESHOLD
SKIP_IF_ALL_CLEAR = True

## Cell 3 — Load the LLM Model

In [3]:
from llama_cpp import Llama

print(f"Loading model from: {MODEL_PATH}")
# llm = Llama(
#     model_path=MODEL_PATH,
#     n_ctx=N_CTX,
#     verbose=False
# )
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=512,          # Idea 2: Dropped from 2048 to 512. Prompts process much faster.
    n_threads=4,        # Idea 1: Forces the Pi 5 to use all 4 physical cores.
    n_batch=256,        # Optimization: Processes the prompt in chunks of 256 tokens.
    verbose=False
)
print("Model loaded and ready.")

Loading model from: ../Models/Qwen2.5-1.5B-Instruct-Q4_K_M.gguf


llama_context: n_ctx_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded and ready.


## Cell 4 — Load RAG Knowledge Base

In [4]:
import json

with open(RAG_PATH, "r") as f:
    rag_data = json.load(f)

# Store chunks in a flat list for easy lookup
CHUNKS = rag_data["chunks"]
print(f"Loaded {len(CHUNKS)} knowledge chunks from '{RAG_PATH}'")

# Preview the chunk types available
types = set(c["type"] for c in CHUNKS)
print(f"Chunk types: {types}")

Loaded 35 knowledge chunks from '../llm/rag_knowledge_base.json'
Chunk types: {'behavioral_guideline', 'mechanism', 'preventative_guideline', 'safety_guideline', 'short_term_intervention', 'risk_explanation'}


## Cell 5 — RAG Retrieval + Helper Functions

**How retrieval works (no vector DB needed):**
The knowledge base is small enough (35 chunks) that we use a **scored keyword match**.
Each chunk gets a relevance score based on how well its `tags` and `type` match
the current payload's conditions. The top-K chunks are returned.

**Scoring logic:**
- Microsleep events → boosts `high-risk`, `micro-sleep`, `stop-driving` chunks
- Drowsy events → boosts `short_term_intervention`, `nap`, `caffeine` chunks
- Worsening trend → boosts `risk_explanation` chunks
- Long microsleep (>5s) → adds extra weight to critical risk chunks

In [5]:
import re


# RAG RETRIEVAL
def score_chunk(chunk: dict, payload: dict) -> int:
    """
    Returns a relevance score for a chunk given the current payload.
    Higher score = more relevant. Zero means not relevant.
    """
    ms    = payload.get("microsleep_events", 0)
    dr    = payload.get("drowsy_events", 0)
    lmf   = payload.get("longest_microsleep_frames", 0)
    trend = payload.get("drowsy_trend", 0)

    tags      = set(chunk.get("tags", []))
    chunk_type = chunk.get("type", "")
    score     = 0

    # Microsleep rules
    if ms > 0:
        if "high-risk" in tags:        score += 3
        if "micro-sleep" in tags:      score += 3
        if "stop-driving" in tags:     score += 2
        if chunk_type == "risk_explanation": score += 1

    # Long microsleep (>5s at 30fps = 150 frames)
    if lmf > 150:
        if "high-risk" in tags:        score += 2   # extra urgency
        if "stop-driving" in tags:     score += 2

    # Drowsy event rules
    if dr > 0:
        if chunk_type == "short_term_intervention": score += 3
        if "nap" in tags:              score += 2
        if "caffeine" in tags:         score += 1
        if "rest-stop" in tags:        score += 1

    # Worsening trend
    if trend > 50:
        if chunk_type == "risk_explanation":  score += 2
        if "sleep-deprivation" in tags: score += 1

    # Preventative content is lower priority during active events
    if chunk_type == "preventative_guideline" and (ms > 0 or dr > 0):
        score = max(0, score - 1)   # slight penalty — not the right moment

    return score


import random

def retrieve_chunks(payload: dict, top_k: int = 2) -> list:
    """
    Scores all chunks against the payload. Grabs the top 5 most relevant, 
    then randomly selects top_k from that pool to provide varied advice.
    """
    # 1. Score all chunks
    scored = [(chunk, score_chunk(chunk, payload)) for chunk in CHUNKS]
    
    # 2. Drop chunks with 0 relevance
    scored = [(c, s) for c, s in scored if s > 0]   
    
    # 3. Sort by highest score first
    scored.sort(key=lambda x: x[1], reverse=True)
    
    # 4. Take the top 5 best candidates
    top_candidates = [c for c, _ in scored[:5]]
    
    # 5. Randomly pick 2 from the top tier to inject variety
    if len(top_candidates) > top_k:
        return random.sample(top_candidates, top_k)
        
    return top_candidates


def chunks_to_context(chunks: list) -> str:
    """
    Formats retrieved chunks into a concise context block for the prompt.
    Each chunk contributes its title + text only (no source URL) to save tokens.
    """
    if not chunks:
        return ""
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[Ref {i}] {chunk['title']}: {chunk['text']}")
    return "\n".join(lines)


# PAYLOAD → NARRATIVE
def is_all_clear(payload: dict) -> bool:
    return (
        payload.get("microsleep_events", 0) == 0
        and payload.get("drowsy_events", 0) == 0
        and abs(payload.get("drowsy_trend", 0)) < 10
    )


def payload_to_narrative(payload: dict) -> str:
    ms    = payload.get("microsleep_events", 0)
    dr    = payload.get("drowsy_events", 0)
    lmf   = payload.get("longest_microsleep_frames", 0)
    ldf   = payload.get("longest_drowsy_frames", 0)
    trend = payload.get("drowsy_trend", 0)
    alert = payload.get("alert_frames", 0)       

    lmf_sec = round(lmf / 30, 1)
    ldf_sec = round(ldf / 30, 1)
    alert_sec = round(alert / 30, 1)              

    trend_desc = "stable"
    if trend > 50:    trend_desc = "rapidly worsening"
    elif trend > 10:  trend_desc = "gradually worsening"
    elif trend < -50: trend_desc = "rapidly improving"
    elif trend < -10: trend_desc = "gradually improving"

    parts = ["In the last 30 seconds:"]
    if ms > 0:
        parts.append(f"The driver had {ms} microsleep episode(s), the longest lasting {lmf_sec}s.")
    if dr > 0:
        parts.append(f"The driver had {dr} drowsy episode(s), the longest lasting {ldf_sec}s.")
    if ms == 0 and dr == 0:
        parts.append("No microsleep or drowsy events were detected.")
    parts.append(f"The driver was alert for approximately {alert_sec}s of this window.")
    parts.append(f"Overall drowsiness trend: {trend_desc} (score: {trend}).")
    return " ".join(parts)


# PROMPT BUILDER (RAG-AWARE)
def build_prompt(narrative: str, context: str) -> str:
    """
    SmolLM2-135M optimized TTS prompt.
    Goal: single natural spoken sentence, no repetition of input text.
    """

    # Build compact input (avoid long RAG dumps triggering copy behavior)
    if context:
        user_msg = (
            "SAFETY NOTES (DO NOT REPEAT):\n"
            f"{context}\n\n"
            "DRIVING STATUS SUMMARY:\n"
            f"{narrative}\n\n"
            "TASK: Say ONE short safety instruction to the driver.\n"
            "RULES:\n"
            "- One sentence only\n"
            "- Do NOT repeat the status text\n"
            "- Do NOT mention numbers, refs, or sources\n"
            "- Do NOT explain\n"
            "- Speak like a calm voice assistant\n"
        )
    else:
        user_msg = (
            "DRIVING STATUS SUMMARY:\n"
            f"{narrative}\n\n"
            "TASK: Say ONE short safety instruction to the driver.\n"
            "RULES:\n"
            "- One sentence only\n"
            "- Do NOT repeat the status text\n"
            "- Do NOT mention numbers or sources\n"
            "- Do NOT explain\n"
            "- Speak like a calm voice assistant\n"
        )

    return (
        "<|im_start|>system\n"
        "You are a real-time driving safety voice assistant. "
        "You NEVER repeat user input. You ALWAYS rephrase in new words. "
        "You output ONLY one spoken sentence.<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

# LLM OUTPUT PARSER
def parse_llm_output(raw: str) -> dict:
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        match = re.search(r"\{.*?\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return {
        "driver_state": "Unknown",
        "risk_level":   "Unknown",
        "message":      "Unable to assess driver state."
    }


# FULL PIPELINE
def run_llm_with_rag(payload: dict) -> dict:
    """
    Full pipeline: payload → RAG retrieval → narrative → enriched prompt → LLM → result.
    """
    # 1. Retrieve relevant chunks
    chunks  = retrieve_chunks(payload)
    context = chunks_to_context(chunks)

    # 2. Build narrative + prompt
    narrative = payload_to_narrative(payload)
    prompt    = build_prompt(narrative, context)

    # 3. Run LLM
    response = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stop=["\n\n", "[INST]"]
    )

    raw_text = response["choices"][0]["text"]
    result   = parse_llm_output(raw_text)

    # Attach which chunks were used (useful for debugging)
    result["_rag_refs"] = [c["id"] for c in chunks]
    return result


print("RAG + helper functions ready.")

RAG + helper functions ready.


## Cell 6 — Main Loop (ZMQ Receiver + RAG + LLM)

Connects to port 5555 and processes each 30-second payload.
Stop with **Kernel → Interrupt**.

In [ ]:
import zmq
import time
import json

# CONFIG
ZMQ_PORT = 5555
SKIP_IF_ALL_CLEAR = True

# PRE-LLM SPOKEN ANNOUNCEMENT
# Maps severity + payload signals to a plain-English state description
# spoken immediately when the payload lands — before the LLM runs.
def get_pre_advisory_announcement(payload: dict, severity: str) -> str:
    ms    = payload.get("microsleep_events", 0)
    dr    = payload.get("drowsy_events", 0)
    trend = payload.get("drowsy_trend", 0)
    alert = payload.get("alert_frames", 0)

    alert_ratio = alert / 900

    if severity == "CRITICAL":
        state = "a critically impaired state"
    elif severity == "HIGH":
        state = "a high-risk drowsy state"
    elif severity == "MODERATE":
        if ms > 0 and dr > 0:
            state = "a moderately drowsy state with microsleep activity"
        elif ms > 0:
            state = "a moderately drowsy state with microsleep events"
        else:
            state = "a moderately drowsy state"
    else:  # LOW
        if trend > 10:
            state = "an early-warning drowsy state with a worsening trend"
        else:
            state = "a mildly drowsy state"

    recovering = alert_ratio >= 0.65 and trend <= 0
    qualifier = ", though showing signs of recovery" if recovering else ""

    return f"Driver observed to be in {state}{qualifier}. Loading AI advisory system."

# DRIVER STATE LOGIC (OPTIMIZED FOR REAL LOG DATA SHAPES)
WINDOW_FRAMES = 900  # ~30s at 30fps
def compute_severity(payload):
    microsleep    = payload.get("microsleep_events", 0)
    drowsy        = payload.get("drowsy_events", 0)
    trend         = payload.get("drowsy_trend", 0)
    ms_frames     = payload.get("longest_microsleep_frames", 0)
    drowsy_frames = payload.get("longest_drowsy_frames", 0)
    alert         = payload.get("alert_frames", 0)

    alert_ratio = min(alert / WINDOW_FRAMES, 1.0)

    if microsleep == 0:       ms_count_score = 0
    elif microsleep <= 2:     ms_count_score = 20
    elif microsleep <= 4:     ms_count_score = 45
    elif microsleep <= 7:     ms_count_score = 60
    else:                     ms_count_score = 75

    if ms_frames == 0:        ms_dur_score = 0
    elif ms_frames < 60:      ms_dur_score = 10
    elif ms_frames < 150:     ms_dur_score = 30
    elif ms_frames < 300:     ms_dur_score = 55
    elif ms_frames < 500:     ms_dur_score = 72
    else:                     ms_dur_score = 88

    if drowsy == 0:           dr_count_score = 0
    elif drowsy <= 2:         dr_count_score = 15
    elif drowsy <= 4:         dr_count_score = 30
    else:                     dr_count_score = 50

    if drowsy_frames == 0:    dr_dur_score = 0
    elif drowsy_frames < 80:  dr_dur_score = 10
    elif drowsy_frames < 200: dr_dur_score = 25
    elif drowsy_frames < 400: dr_dur_score = 42
    else:                     dr_dur_score = 65

    if trend <= 0:            trend_score = 0
    elif trend < 50:          trend_score = 10
    elif trend < 150:         trend_score = 25
    elif trend < 300:         trend_score = 40
    else:                     trend_score = 55

    raw_score = (
        ms_count_score * 0.25 +
        ms_dur_score   * 0.30 +
        dr_count_score * 0.12 +
        dr_dur_score   * 0.13 +
        trend_score    * 0.20
    )

    if alert_ratio >= 0.80:   modifier = -6
    elif alert_ratio >= 0.60: modifier = -3
    elif alert_ratio <= 0.10: modifier = +12
    elif alert_ratio <= 0.25: modifier = +6
    else:                     modifier = 0

    score = raw_score + modifier

    if score >= 55: return "CRITICAL"
    if score >= 35: return "HIGH"
    if score >= 15: return "MODERATE"
    return "LOW"


# alert_frames now acts as the positive confirmation signal.
# A window needs BOTH zero bad events AND meaningful alert time.
# At 30fps, a 30s window = ~900 frames total.
def is_all_clear(payload):
    alert = payload.get("alert_frames", 0)
    return (
        payload.get("microsleep_events", 0) == 0
        and payload.get("drowsy_events", 0) == 0
        and payload.get("longest_microsleep_frames", 0) < 15
        and payload.get("longest_drowsy_frames", 0) < 15
        and payload.get("drowsy_trend", 0) <= 0
        and alert >= 600
    )

# ── build_prompt ────────────────────────────────────────────────
# Severity context injected into the prompt so the LLM understands
# *why* the urgency level was assigned, not just what it is.
SEVERITY_CONTEXT = {
    "LOW":      "The driver shows minor early-warning signs. A gentle, non-alarming nudge is appropriate.",
    "MODERATE": "The driver shows consistent drowsiness signals. A clear, firm advisory is needed.",
    "HIGH":     "The driver is significantly impaired. Be direct and urgent — recommend immediate action.",
    "CRITICAL": "The driver is in a dangerous state. This is an emergency advisory. Be brief, firm, and action-focused.",
}

def build_prompt(narrative: str, context: str, severity: str, alert_frames: int = 0) -> str:
    severity_guidance = SEVERITY_CONTEXT.get(severity, "")

    alert_ratio = alert_frames / WINDOW_FRAMES
    if alert_ratio >= 0.80:
        alert_context = "The driver is mostly alert and appears to be recovering."
    elif alert_ratio >= 0.50:
        alert_context = "The driver had mixed alertness — some recovery but still at risk."
    elif alert_ratio >= 0.20:
        alert_context = "The driver was alert for only a short portion of the window."
    else:
        alert_context = "The driver was barely alert this window. Treat this as near-critical regardless of event counts."

    system_prompt = (
        "You are an AI vehicle safety co-pilot speaking directly to a driver over car speakers. "
        "Your only output is a single valid JSON object — no markdown, no preamble, no trailing text.\n\n"
        "Required schema:\n"
        "{\n"
        '  "actionable_tip": "Two to three natural spoken sentences directed at the driver.",\n'
        '  "estimated_urgency": "LOW | MODERATE | HIGH | CRITICAL",\n'
        '  "intervention_type": "One word: Nap | Caffeine | Focus | Rest | Pullover | Stimulate"\n'
        "}\n\n"
        "Rules for actionable_tip:\n"
        "- Exactly 2–3 sentences. No more.\n"
        "- Spoken language only — no lists, no symbols, no clinical jargon.\n"
        "- Match tone to urgency: calm for LOW, firm and brief for HIGH/CRITICAL.\n"
        "- Give one concrete action, not general advice."
    )

    rag_block = (
        f"RELEVANT SAFETY GUIDANCE (use this to ground your tip):\n{context}\n\n"
        if context else ""
    )

    user_msg = (
        f"DRIVER STATUS:\n{narrative}\n\n"
        f"ALERTNESS THIS WINDOW: {alert_context}\n\n"
        f"{rag_block}"
        f"SEVERITY: {severity} — {severity_guidance}\n\n"
        "Generate the JSON object now."
    )

    return (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        "<|im_start|>assistant\n{"
    )

# MAIN LOOP
context = zmq.Context()
# socket  = context.socket(zmq.PULL) <-------------------------------------------------------------------------------------------
# socket.connect(f"tcp://localhost:{ZMQ_PORT}")
socket = context.socket(zmq.SUB)
socket.connect(f"tcp://localhost:{ZMQ_PORT}")
socket.setsockopt_string(zmq.SUBSCRIBE, "")

print(f"LLM Advisor (TTS JSON mode) listening on port {ZMQ_PORT}...")
print("Waiting for payloads...\n")

try:
    while True:
        payload = socket.recv_json()
        
        # Start generation timer immediately when a payload lands
        batch_start_time = time.time()
        
        print(f"[RECEIVED] {payload}")

        # Skip clean state
        if SKIP_IF_ALL_CLEAR and is_all_clear(payload):
            print("[SKIP] Driver is fully alert\n")
            continue
        severity = compute_severity(payload)
        announcement = get_pre_advisory_announcement(payload, severity)
        print(f"[ANNOUNCEMENT]   → {announcement}")
        chunks = retrieve_chunks(payload)
        context_text = chunks_to_context(chunks) if chunks else ""
        narrative = payload_to_narrative(payload)
        prompt = build_prompt(narrative, context_text, severity, payload.get("alert_frames", 0))

        llm_t0 = time.time()
        response = llm(
            prompt,
            max_tokens= 600,  # Expanded context to gracefully capture multiple sentences
            temperature=0.3, # Keeps JSON structure stable while letting prose breathe
            stop=["<|im_end|>", "\n\n\n"]
        )
        llm_elapsed = time.time() - llm_t0

        # Reconstruct the valid JSON string by adding the opening brace back
        raw_json_str = "{" + response["choices"][0]["text"].strip()
        
        # Safe JSON parsing
        try:
            parsed_response = json.loads(raw_json_str)
            message = parsed_response.get("actionable_tip", "Please stay alert.")
            urgency = parsed_response.get("estimated_urgency", severity)
            intervention = parsed_response.get("intervention_type", "General")
        except Exception as e:
            print(f"[ERROR] Failed to parse JSON: {e}")
            print(f"Raw output was: {raw_json_str}")
            # Fallback values
            message = "I noticed you are getting drowsy. Please find a safe spot to pull over and rest for a bit."
            urgency = severity
            intervention = "Fallback"

        # Calculate final pipeline latency
        total_batch_pipeline_time = time.time() - batch_start_time

        # =========================
        # OUTPUT & PERFORMANCE METRICS
        # =========================
        print(f"[LLM ONLY TIME]  → {llm_elapsed:.2f}s")
        print(f"[TOTAL PIPELINE] → {total_batch_pipeline_time:.2f}s (From Payload to Output)")
        print(f"[METADATA]       → Urgency: {urgency} | Type: {intervention}")
        print(f"[TTS READY]      → {message}")

        rag_refs = [c["id"] for c in chunks] if chunks else []
        print(f"[REFS USED]      → {rag_refs}\n")

except KeyboardInterrupt:
    print("\nStopped by user.")

finally:
    socket.close()
    context.term()
    print("ZMQ connection closed.")

LLM Advisor (TTS JSON mode) listening on port 5555...
Waiting for payloads...

[RECEIVED] {'microsleep_events': 2, 'drowsy_events': 6, 'longest_microsleep_frames': 89, 'longest_drowsy_frames': 107, 'drowsy_trend': 39, 'alert_frames': 145}
[ANNOUNCEMENT]   → Driver observed to be in a moderately drowsy state with microsleep activity. Loading AI advisory system.
[LLM ONLY TIME]  → 4.65s
[TOTAL PIPELINE] → 4.65s (From Payload to Output)
[METADATA]       → Urgency: HIGH | Type: Pullover
[TTS READY]      → Pull over immediately and rest for at least 20 minutes. Avoid driving until fully refreshed.
[REFS USED]      → ['sleepfoundation_003', 'nhtsa_intervention_002']

[RECEIVED] {'microsleep_events': 2, 'drowsy_events': 0, 'longest_microsleep_frames': 129, 'longest_drowsy_frames': 0, 'drowsy_trend': -5, 'alert_frames': 571}
[ANNOUNCEMENT]   → Driver observed to be in a mildly drowsy state. Loading AI advisory system.
[LLM ONLY TIME]  → 10.89s
[TOTAL PIPELINE] → 10.89s (From Payload to Output)

## Cell 7 — Debug: See Raw LLM Output <- Cell 7 and 8 Only for Debuging!!!
Run this first to see exactly what the model is generating before JSON parsing.

## Cell 8 — (Optional) Quick Test Without ZMQ

Verify RAG retrieval and LLM output without needing the full camera pipeline.
Also prints which knowledge chunks were selected for each scenario.